# Fine-tune YOLO11 on plastic / can

1. *Add Input* → dataset made from `plasticcan_clean.zip`.
2. *Settings* → **GPU T4**, **Internet On**.
3. *Run All* (~5 min).
4. Download `/kaggle/working/trash_yolo11.pt`, put it next to `scan_trash.py`.

In [ ]:
!pip install -q "ultralytics>=8.3.0"

In [ ]:
import glob, os, shutil
import yaml
from ultralytics import YOLO

BASE_MODEL = "yolo11s.pt"   # COCO-pretrained, downloaded automatically ("yolo11n.pt" = smaller/faster)
EPOCHS = 100                # ~84 images -> each epoch is a few seconds, 100 epochs ~5 min on a T4
PATIENCE = 40               # only 8 val images -> val mAP is noisy, don't stop too early
IMGSZ = 640                 # letterbox resize of the whole frame (1920x1080 -> 640x360 + padding), NOT a crop
WORK = "/kaggle/working"

# Roboflow's data.yaml uses ../train/images paths -> rewrite with the absolute dataset path
src_yaml = glob.glob("/kaggle/input/**/data.yaml", recursive=True)[0]
src = os.path.dirname(src_yaml)
cfg = yaml.safe_load(open(src_yaml))
data = {"path": src, "train": "train/images", "val": "valid/images", "names": cfg["names"]}
if os.path.isdir(f"{src}/test/images"):
    data["test"] = "test/images"
DATA_YAML = f"{WORK}/data.yaml"
yaml.safe_dump(data, open(DATA_YAML, "w"))
print(open(DATA_YAML).read())

In [ ]:
model = YOLO(BASE_MODEL)
model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMGSZ,
    # Tiny dataset fixes:
    batch=8,
    nbs=8,              # no gradient accumulation: default nbs=64 meant only ~1 weight update per epoch
    optimizer="AdamW",
    lr0=0.0005,         # explicit low LR for fine-tuning (the previous run diverged)
    warmup_epochs=3,
    freeze=10,          # keep the pretrained backbone fixed -> stable, less overfitting
    amp=False,          # rule out fp16 overflow (val loss exploded to ~300); tiny dataset, speed is fine
    device=0,
    cache="ram",
    workers=2,
    # Roboflow already added rotations; just flips + light changes here
    flipud=0.5,
    fliplr=0.5,
    hsv_v=0.4,
    close_mosaic=15,    # last epochs without mosaic -> matches real single-object frames
    project=f"{WORK}/runs",
    name="trash",
    exist_ok=True,
)
shutil.copy2(f"{WORK}/runs/trash/weights/best.pt", f"{WORK}/trash_yolo11.pt")
print("Saved /kaggle/working/trash_yolo11.pt")

In [ ]:
from IPython.display import Image, display
final = YOLO(f"{WORK}/trash_yolo11.pt")
print("Classes:", final.names)
display(Image(f"{WORK}/runs/trash/results.png", width=900))
final.predict(sorted(glob.glob(f"{src}/valid/images/*"))[:4], save=True,
              project=f"{WORK}/runs", name="preview", exist_ok=True)
for p in sorted(glob.glob(f"{WORK}/runs/preview/*"))[:4]:
    display(Image(p, width=480))